[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnCrabs/GeoEducationGR/blob/main/ProjectExamples/Python/01_Geometria_Point_Line_Polygon.ipynb)

# Εφαρμογή 1: Γεωμετρικές οντότητες στο επίπεδο

Σε αυτή την εφαρμογή φτιάχνουμε από την αρχή τρεις βασικές κλάσεις για γεωμετρία στο επίπεδο:

| Κλάση | Τι αναπαριστά |
|---|---|
| `Point2D` | Ένα σημείο με συντεταγμένες $(X, Y)$ |
| `Line2D` | Ένα ευθύγραμμο τμήμα που ορίζεται από δύο σημεία |
| `Polygon` | Ένα πολύγωνο που ορίζεται από μια σειρά κορυφών |

Κάθε κλάση έχει:
- **υπολογισμούς** (απόσταση, αζιμούθιο, μήκος, εμβαδόν, κέντρο βάρους κ.ά.),
- μια συνάρτηση **`describe()`** που επιστρέφει μια πλήρη περιγραφή της οντότητας σε κείμενο,
- μια συνάρτηση **`plot()`** για να τη σχεδιάσουμε.

Χρησιμοποιούμε μόνο την τυπική βιβλιοθήκη της Python (`math`) και το `matplotlib`, που υπάρχει ήδη στο Google Colab, οπότε δεν χρειάζεται καμία εγκατάσταση.

## 0. Σύμβαση συντεταγμένων και αζιμουθίου

Ακολουθούμε τη σύμβαση της Τοπογραφίας:

- Ο άξονας **Y** δείχνει τον **Βορρά** και ο άξονας **X** την **Ανατολή**.
- Το **αζιμούθιο** $\alpha_{AB}$ της διεύθυνσης $A \rightarrow B$ μετριέται από τον Βορρά (άξονας Y), **δεξιόστροφα**:

$$
\alpha_{AB} = \operatorname{atan2}(\Delta X_{AB},\ \Delta Y_{AB}), \qquad \Delta X_{AB} = X_B - X_A,\quad \Delta Y_{AB} = Y_B - Y_A
$$

- Το αποτέλεσμα φέρνεται στο διάστημα $[0, 400)$ **grad** (ή $[0^\circ, 360^\circ)$).
- Η **απόσταση** είναι $S_{AB} = \sqrt{\Delta X_{AB}^2 + \Delta Y_{AB}^2}$.

Προσοχή: η `math.atan2(y, x)` της Python δέχεται πρώτα το «y». Εδώ της δίνουμε πρώτα το $\Delta X$, επειδή η γωνία μετριέται από τον άξονα Y.

In [ ]:
import math
import matplotlib.pyplot as plt

# Ανοχή για συγκρίσεις αριθμών κινητής υποδιαστολής
# (π.χ. 0.1 + 0.2 δεν είναι ακριβώς 0.3 στον υπολογιστή)
TOL = 1e-9

# Μετατροπές γωνιών
RAD_TO_GRAD = 200.0 / math.pi
RAD_TO_DEG = 180.0 / math.pi


def convert_angle(angle_rad, unit="grad"):
    # Μετατρέπει γωνία από ακτίνια στη μονάδα που ζητάμε: 'grad', 'deg' ή 'rad'
    if unit == "grad":
        return angle_rad * RAD_TO_GRAD
    if unit == "deg":
        return angle_rad * RAD_TO_DEG
    if unit == "rad":
        return angle_rad
    raise ValueError(f"Άγνωστη μονάδα γωνίας: {unit!r}. Επιτρέπονται 'grad', 'deg', 'rad'.")


print("Έτοιμο!")

## 1. Η κλάση `Point2D`

Ένα σημείο έχει συντεταγμένες $X, Y$ και προαιρετικά ένα **όνομα** (π.χ. `"A"`, `"Σ1"`).

Βασικές μέθοδοι:

| Μέθοδος | Τι κάνει |
|---|---|
| `distance_to(p)` | Απόσταση από άλλο σημείο |
| `delta_to(p)` | Διαφορές $\Delta X, \Delta Y$ προς άλλο σημείο |
| `azimuth_to(p, unit)` | Αζιμούθιο προς άλλο σημείο |
| `midpoint(p)` | Μέσο του τμήματος με άλλο σημείο |
| `translate(dx, dy)` | Νέο σημείο μετατοπισμένο κατά $dx, dy$ |
| `rotate(angle, center)` | Νέο σημείο στραμμένο γύρω από ένα κέντρο |
| `quadrant()` | Σε ποιο τεταρτημόριο βρίσκεται |
| `describe()` | Πλήρης περιγραφή σε κείμενο |
| `plot(ax)` | Σχεδίαση |

Οι **ειδικές μέθοδοι** (dunder methods) `__repr__`, `__str__`, `__eq__` και `__iter__` κάνουν την κλάση να συμπεριφέρεται «φυσικά» στην Python: σωστή εκτύπωση, σύγκριση με `==` και αποσυσκευασία `x, y = p`.

In [ ]:
class Point2D:
    """Σημείο στο επίπεδο με συντεταγμένες (X, Y) και προαιρετικό όνομα."""

    def __init__(self, x, y, name=None):
        self.x = float(x)
        self.y = float(y)
        self.name = name

    # ------------------------------------------------ Ειδικές μέθοδοι
    def __repr__(self):
        # Τεχνική αναπαράσταση: από αυτή μπορεί να ξαναφτιαχτεί το αντικείμενο
        return f"Point2D(x={self.x}, y={self.y}, name={self.name!r})"

    def __str__(self):
        # Φιλική αναπαράσταση για εκτύπωση με print()
        label = self.name if self.name else "P"
        return f"{label}({self.x:.3f}, {self.y:.3f})"

    def __eq__(self, other):
        # Δύο σημεία είναι ίσα αν οι συντεταγμένες τους συμπίπτουν (με ανοχή TOL)
        if not isinstance(other, Point2D):
            return NotImplemented
        return (math.isclose(self.x, other.x, abs_tol=TOL)
                and math.isclose(self.y, other.y, abs_tol=TOL))

    def __iter__(self):
        # Επιτρέπει: x, y = p
        yield self.x
        yield self.y

    # ------------------------------------------------ Βασικές πληροφορίες
    def to_tuple(self):
        return (self.x, self.y)

    def quadrant(self):
        """Επιστρέφει το τεταρτημόριο (1-4) ή περιγραφή αν το σημείο είναι πάνω σε άξονα."""
        if math.isclose(self.x, 0, abs_tol=TOL) and math.isclose(self.y, 0, abs_tol=TOL):
            return "αρχή των αξόνων"
        if math.isclose(self.x, 0, abs_tol=TOL):
            return "πάνω στον άξονα Y"
        if math.isclose(self.y, 0, abs_tol=TOL):
            return "πάνω στον άξονα X"
        if self.x > 0 and self.y > 0:
            return 1
        if self.x < 0 and self.y > 0:
            return 2
        if self.x < 0 and self.y < 0:
            return 3
        return 4

    # ------------------------------------------------ Σχέσεις με άλλο σημείο
    def delta_to(self, other):
        """Επιστρέφει (ΔX, ΔY) από αυτό το σημείο προς το other."""
        return (other.x - self.x, other.y - self.y)

    def distance_to(self, other):
        """Ευκλείδεια απόσταση S = sqrt(ΔX² + ΔY²)."""
        dx, dy = self.delta_to(other)
        return math.hypot(dx, dy)

    def azimuth_to(self, other, unit="grad"):
        """Αζιμούθιο από τον Βορρά (άξονας Y), δεξιόστροφα.

        unit: 'grad' (0-400), 'deg' (0-360) ή 'rad' (0-2π)
        """
        dx, dy = self.delta_to(other)
        if math.isclose(dx, 0, abs_tol=TOL) and math.isclose(dy, 0, abs_tol=TOL):
            raise ValueError("Τα δύο σημεία συμπίπτουν: το αζιμούθιο δεν ορίζεται.")
        a = math.atan2(dx, dy)          # Προσοχή: πρώτα ΔX, μετά ΔY
        if a < 0:
            a += 2 * math.pi            # Φέρνουμε τη γωνία στο [0, 2π)
        return convert_angle(a, unit)

    def midpoint(self, other, name=None):
        """Μέσο του τμήματος που ενώνει τα δύο σημεία."""
        return Point2D((self.x + other.x) / 2, (self.y + other.y) / 2, name)

    # ------------------------------------------------ Μετασχηματισμοί (επιστρέφουν ΝΕΟ σημείο)
    def translate(self, dx, dy, name=None):
        """Μετατόπιση κατά (dx, dy)."""
        return Point2D(self.x + dx, self.y + dy, name or self.name)

    def rotate(self, angle_deg, center=None, name=None):
        """Στροφή κατά angle_deg μοίρες αριστερόστροφα (μαθηματική φορά) γύρω από το center.

        Αν δεν δοθεί center, η στροφή γίνεται γύρω από την αρχή (0, 0).
        """
        cx, cy = (center.x, center.y) if center else (0.0, 0.0)
        t = math.radians(angle_deg)
        x0, y0 = self.x - cx, self.y - cy
        xr = x0 * math.cos(t) - y0 * math.sin(t)
        yr = x0 * math.sin(t) + y0 * math.cos(t)
        return Point2D(xr + cx, yr + cy, name or self.name)

    # ------------------------------------------------ Περιγραφή και σχεδίαση
    def describe(self):
        """Επιστρέφει πλήρη περιγραφή του σημείου σε κείμενο."""
        origin = Point2D(0, 0)
        lines = [
            f"Σημείο: {self.name if self.name else '(χωρίς όνομα)'}",
            f"  X = {self.x:.3f}",
            f"  Y = {self.y:.3f}",
            f"  Τεταρτημόριο: {self.quadrant()}",
            f"  Απόσταση από την αρχή (0, 0): {self.distance_to(origin):.3f}",
        ]
        if self != origin:
            lines.append(f"  Αζιμούθιο από την αρχή: {origin.azimuth_to(self):.4f} grad"
                         f" ({origin.azimuth_to(self, 'deg'):.4f}°)")
        return "\n".join(lines)

    def plot(self, ax=None, color="tab:red", show_label=True, **kwargs):
        """Σχεδιάζει το σημείο. Επιστρέφει τον άξονα (ax) για να προσθέσουμε κι άλλα στοιχεία."""
        if ax is None:
            _, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(self.x, self.y, color=color, zorder=3, **kwargs)
        if show_label:
            ax.annotate(str(self), (self.x, self.y), textcoords="offset points",
                        xytext=(5, 5), fontsize=9)
        ax.set_aspect("equal")
        ax.grid(True, linestyle=":")
        ax.set_xlabel("X (Ανατολή)")
        ax.set_ylabel("Y (Βορράς)")
        return ax

### Παραδείγματα με `Point2D`

In [ ]:
A = Point2D(100.0, 200.0, "A")
B = Point2D(150.0, 260.0, "B")

print(A)              # χρησιμοποιεί την __str__
print(repr(A))        # χρησιμοποιεί την __repr__
print()
print(A.describe())

In [ ]:
print(f"ΔX, ΔY από A προς B : {A.delta_to(B)}")
print(f"Απόσταση S_AB       : {A.distance_to(B):.3f}")
print(f"Αζιμούθιο α_AB      : {A.azimuth_to(B):.4f} grad = {A.azimuth_to(B, 'deg'):.4f}°")
print(f"Αζιμούθιο α_BA      : {B.azimuth_to(A):.4f} grad  (διαφέρει κατά 200 grad)")
print(f"Μέσο του AB         : {A.midpoint(B, 'M')}")
print(f"A == Point2D(100, 200)? {A == Point2D(100, 200)}")

x, y = A              # αποσυσκευασία χάρη στην __iter__
print(f"x = {x}, y = {y}")

In [ ]:
# Μετασχηματισμοί: επιστρέφουν νέο σημείο, το αρχικό δεν αλλάζει
A_moved = A.translate(10, -20, "A'")
A_rot = A.rotate(90, center=B, name="A''")

ax = A.plot()
B.plot(ax, color="tab:blue")
A_moved.plot(ax, color="tab:green")
A_rot.plot(ax, color="tab:purple")
ax.set_title("Σημεία και μετασχηματισμοί")
plt.show()

## 2. Η κλάση `Line2D`

Η `Line2D` αναπαριστά ένα **ευθύγραμμο τμήμα** από το σημείο `start` στο σημείο `end`. Έχει φορά, άρα το $AB$ έχει διαφορετικό αζιμούθιο από το $BA$.

Βασικές μέθοδοι:

| Μέθοδος | Τι κάνει |
|---|---|
| `dx`, `dy`, `length` | $\Delta X$, $\Delta Y$, μήκος |
| `azimuth(unit)` | Αζιμούθιο της διεύθυνσης start → end |
| `angle_with_x_axis(unit)` | Γωνία με τον άξονα X (γωνία ύψους), αριστερόστροφα |
| `slope()` | Κλίση $\Delta Y / \Delta X$ (άπειρο για κατακόρυφη) |
| `equation()` | Συντελεστές της ευθείας $aX + bY + c = 0$ |
| `midpoint()`, `point_at(t)` | Μέσο και σημείο σε ποσοστό $t \in [0, 1]$ του μήκους |
| `contains_point(p)` | Αν ένα σημείο βρίσκεται πάνω στο τμήμα |
| `distance_to_point(p)` | Ελάχιστη απόσταση σημείου από το τμήμα |
| `is_parallel(l)`, `is_perpendicular(l)` | Παραλληλία και καθετότητα |
| `intersection(l)` | Σημείο τομής με άλλο τμήμα (ή `None`) |
| `reversed()` | Το ίδιο τμήμα με αντίθετη φορά |
| `describe()`, `plot(ax)` | Περιγραφή και σχεδίαση |

Για την τομή δύο τμημάτων $P + t\,\mathbf{r}$ και $Q + u\,\mathbf{s}$ χρησιμοποιούμε το εξωτερικό γινόμενο $\mathbf{a} \times \mathbf{b} = a_x b_y - a_y b_x$:

$$
t = \frac{(Q - P) \times \mathbf{s}}{\mathbf{r} \times \mathbf{s}}, \qquad
u = \frac{(Q - P) \times \mathbf{r}}{\mathbf{r} \times \mathbf{s}}
$$

Τα τμήματα τέμνονται αν $0 \le t \le 1$ και $0 \le u \le 1$.

In [ ]:
def cross(ax_, ay_, bx_, by_):
    """Εξωτερικό γινόμενο 2D διανυσμάτων: a × b = ax·by − ay·bx."""
    return ax_ * by_ - ay_ * bx_


class Line2D:
    """Ευθύγραμμο τμήμα από το σημείο start στο σημείο end."""

    def __init__(self, start, end, name=None):
        if not isinstance(start, Point2D) or not isinstance(end, Point2D):
            raise TypeError("Τα άκρα του τμήματος πρέπει να είναι αντικείμενα Point2D.")
        if start == end:
            raise ValueError("Τα άκρα συμπίπτουν: το τμήμα θα είχε μηδενικό μήκος.")
        self.start = start
        self.end = end
        # Αν δεν δοθεί όνομα αλλά τα σημεία έχουν, το όνομα γίνεται π.χ. "AB"
        if name is None and start.name and end.name:
            name = f"{start.name}{end.name}"
        self.name = name

    # ------------------------------------------------ Ειδικές μέθοδοι
    def __repr__(self):
        return f"Line2D(start={self.start!r}, end={self.end!r}, name={self.name!r})"

    def __str__(self):
        label = self.name if self.name else "L"
        return f"{label}: {self.start} → {self.end}"

    def __eq__(self, other):
        # Ίσα αν έχουν τα ίδια άκρα με την ίδια φορά
        if not isinstance(other, Line2D):
            return NotImplemented
        return self.start == other.start and self.end == other.end

    # ------------------------------------------------ Βασικά μεγέθη
    @property
    def dx(self):
        return self.end.x - self.start.x

    @property
    def dy(self):
        return self.end.y - self.start.y

    @property
    def length(self):
        return math.hypot(self.dx, self.dy)

    def azimuth(self, unit="grad"):
        """Αζιμούθιο της διεύθυνσης start → end."""
        return self.start.azimuth_to(self.end, unit)

    def slope(self):
        """Κλίση ΔY/ΔX. Για κατακόρυφο τμήμα επιστρέφει math.inf."""
        if math.isclose(self.dx, 0, abs_tol=TOL):
            return math.inf
        return self.dy / self.dx

    def angle_with_x_axis(self, unit="grad"):
        """Γωνία ύψους: γωνία με τον άξονα X, αριστερόστροφα (μαθηματική φορά), στο [0, 2π)."""
        a = math.atan2(self.dy, self.dx)
        if a < 0:
            a += 2 * math.pi
        return convert_angle(a, unit)

    def equation(self):
        """Συντελεστές (a, b, c) της ευθείας aX + bY + c = 0 που περνά από το τμήμα."""
        a = self.dy
        b = -self.dx
        c = -(a * self.start.x + b * self.start.y)
        return a + 0.0, b + 0.0, c + 0.0    # + 0.0: μετατρέπει το -0.0 σε 0.0

    def equation_str(self):
        a, b, c = self.equation()
        return f"{a:.3f}·X {b:+.3f}·Y {c:+.3f} = 0"

    # ------------------------------------------------ Σημεία πάνω στο τμήμα
    def midpoint(self, name=None):
        return self.start.midpoint(self.end, name)

    def point_at(self, t, name=None):
        """Σημείο σε ποσοστό t του μήκους (t=0 → start, t=1 → end)."""
        return Point2D(self.start.x + t * self.dx, self.start.y + t * self.dy, name)

    def reversed(self):
        """Το ίδιο τμήμα με αντίθετη φορά (end → start)."""
        name = self.name[::-1] if self.name else None
        return Line2D(self.end, self.start, name)

    # ------------------------------------------------ Σχέσεις με σημεία
    def _projection_t(self, p):
        # Παράμετρος t της ορθής προβολής του p πάνω στην ευθεία του τμήματος
        return ((p.x - self.start.x) * self.dx + (p.y - self.start.y) * self.dy) / self.length ** 2

    def distance_to_point(self, p):
        """Ελάχιστη απόσταση του σημείου p από το τμήμα (όχι από την άπειρη ευθεία)."""
        t = min(1.0, max(0.0, self._projection_t(p)))
        return p.distance_to(self.point_at(t))

    def contains_point(self, p, tol=1e-6):
        """True αν το σημείο p βρίσκεται πάνω στο τμήμα (με ανοχή tol)."""
        return self.distance_to_point(p) <= tol

    # ------------------------------------------------ Σχέσεις με άλλο τμήμα
    def is_parallel(self, other):
        return math.isclose(cross(self.dx, self.dy, other.dx, other.dy), 0,
                            abs_tol=TOL * self.length * other.length)

    def is_perpendicular(self, other):
        dot = self.dx * other.dx + self.dy * other.dy
        return math.isclose(dot, 0, abs_tol=TOL * self.length * other.length)

    def intersection(self, other, name=None):
        """Σημείο τομής δύο τμημάτων ή None αν δεν τέμνονται.

        Για παράλληλα (ή συνευθειακά) τμήματα επιστρέφει None.
        """
        rx, ry = self.dx, self.dy
        sx, sy = other.dx, other.dy
        denom = cross(rx, ry, sx, sy)
        if math.isclose(denom, 0, abs_tol=TOL * self.length * other.length):
            return None
        qpx = other.start.x - self.start.x
        qpy = other.start.y - self.start.y
        t = cross(qpx, qpy, sx, sy) / denom
        u = cross(qpx, qpy, rx, ry) / denom
        eps = 1e-12
        if -eps <= t <= 1 + eps and -eps <= u <= 1 + eps:
            return self.point_at(t, name)
        return None

    # ------------------------------------------------ Περιγραφή και σχεδίαση
    def describe(self):
        """Επιστρέφει πλήρη περιγραφή του τμήματος σε κείμενο."""
        slope = self.slope()
        slope_txt = "άπειρη (κατακόρυφο τμήμα)" if math.isinf(slope) else f"{slope:.4f}"
        lines = [
            f"Ευθύγραμμο τμήμα: {self.name if self.name else '(χωρίς όνομα)'}",
            f"  Αρχή : {self.start}",
            f"  Τέλος: {self.end}",
            f"  ΔX = {self.dx:.3f},  ΔY = {self.dy:.3f}",
            f"  Μήκος: {self.length:.3f}",
            f"  Αζιμούθιο: {self.azimuth():.4f} grad ({self.azimuth('deg'):.4f}°)",
            f"  Γωνία με τον άξονα X (γωνία ύψους): {self.angle_with_x_axis('grad'):.4f} grad"
            f" ({self.angle_with_x_axis('deg'):.4f}°)",
            f"  Κλίση: {slope_txt}",
            f"  Εξίσωση ευθείας: {self.equation_str()}",
            f"  Μέσο: {self.midpoint('M')}",
        ]
        return "\n".join(lines)

    def plot(self, ax=None, color="tab:blue", show_points=True, show_label=True, arrow=False, **kwargs):
        """Σχεδιάζει το τμήμα. Με arrow=True δείχνει και τη φορά του."""
        if ax is None:
            _, ax = plt.subplots(figsize=(5, 5))
        if arrow:
            ax.annotate("", xy=self.end.to_tuple(), xytext=self.start.to_tuple(),
                        arrowprops=dict(arrowstyle="->", color=color, lw=1.5))
        else:
            ax.plot([self.start.x, self.end.x], [self.start.y, self.end.y], color=color, **kwargs)
        if show_points:
            self.start.plot(ax, color=color, show_label=show_label)
            self.end.plot(ax, color=color, show_label=show_label)
        if show_label and self.name:
            m = self.midpoint()
            ax.annotate(self.name, m.to_tuple(), textcoords="offset points",
                        xytext=(-10, -12), fontsize=9, color=color)
        ax.set_aspect("equal")
        ax.grid(True, linestyle=":")
        ax.set_xlabel("X (Ανατολή)")
        ax.set_ylabel("Y (Βορράς)")
        return ax

### Παραδείγματα με `Line2D`

In [ ]:
AB = Line2D(A, B)          # το όνομα "AB" προκύπτει αυτόματα από τα σημεία
print(AB)
print()
print(AB.describe())

In [ ]:
C = Point2D(100.0, 260.0, "C")
D = Point2D(160.0, 190.0, "D")
CD = Line2D(C, D)

P = AB.intersection(CD, name="T")
print(f"Τομή AB και CD          : {P}")
print(f"Το T ανήκει στο AB;      {AB.contains_point(P)}")
print(f"Απόσταση C από το AB     : {AB.distance_to_point(C):.3f}")
print(f"AB παράλληλο στο CD;     {AB.is_parallel(CD)}")
print(f"Σημείο στο 25% του AB    : {AB.point_at(0.25)}")
print(f"Αντίστροφο τμήμα         : {AB.reversed()}  (αζιμούθιο {AB.reversed().azimuth():.4f} grad)")

ax = AB.plot(arrow=True)
CD.plot(ax, color="tab:orange", arrow=True)
P.plot(ax, color="black")
ax.set_title("Τομή δύο ευθύγραμμων τμημάτων")
plt.show()

## 3. Η κλάση `Polygon`

Ένα πολύγωνο ορίζεται από μια **λίστα κορυφών** (`Point2D`) με τη σειρά που τις διατρέχουμε. Το πολύγωνο κλείνει αυτόματα: η τελευταία κορυφή ενώνεται με την πρώτη. Αν η πρώτη κορυφή επαναληφθεί στο τέλος, η επανάληψη αφαιρείται.

Βασικές μέθοδοι:

| Μέθοδος | Τι κάνει |
|---|---|
| `edges()` | Λίστα με τις πλευρές ως `Line2D` |
| `perimeter()` | Περίμετρος |
| `signed_area()`, `area()` | Προσημασμένο και απόλυτο εμβαδόν (τύπος του Gauss) |
| `orientation()` | Φορά διαγραφής των κορυφών (δεξιόστροφη ή αριστερόστροφη) |
| `centroid()` | Κέντρο βάρους |
| `bounding_box()` | Ελάχιστο ορθογώνιο που το περικλείει |
| `interior_angles(unit)` | Εσωτερικές γωνίες σε κάθε κορυφή |
| `is_convex()` | Αν είναι κυρτό |
| `is_simple()` | Αν είναι απλό (οι πλευρές του δεν αυτοτέμνονται) |
| `contains_point(p)` | Αν ένα σημείο είναι μέσα στο πολύγωνο |
| `describe()`, `plot(ax)` | Περιγραφή και σχεδίαση |

**Εμβαδόν με τον τύπο του Gauss (shoelace):**

$$
E = \frac{1}{2} \sum_{i=1}^{n} \left( X_i\,Y_{i+1} - X_{i+1}\,Y_i \right)
$$

Αν $E > 0$, οι κορυφές διατρέχονται **αριστερόστροφα**. Αν $E < 0$, **δεξιόστροφα**. Το εμβαδόν του πολυγώνου είναι $|E|$.

**Κέντρο βάρους:**

$$
X_G = \frac{1}{6E} \sum_{i=1}^{n} (X_i + X_{i+1})(X_i Y_{i+1} - X_{i+1} Y_i), \qquad
Y_G = \frac{1}{6E} \sum_{i=1}^{n} (Y_i + Y_{i+1})(X_i Y_{i+1} - X_{i+1} Y_i)
$$

**Σημείο μέσα σε πολύγωνο (ray casting):** από το σημείο «ρίχνουμε» μια οριζόντια ημιευθεία και μετράμε πόσες πλευρές τέμνει. Αν τις τέμνει περιττό αριθμό φορών, το σημείο είναι μέσα.

In [ ]:
POLYGON_NAMES = {3: "Τρίγωνο", 4: "Τετράπλευρο", 5: "Πεντάγωνο", 6: "Εξάγωνο",
                 7: "Επτάγωνο", 8: "Οκτάγωνο", 9: "Εννεάγωνο", 10: "Δεκάγωνο"}


class Polygon:
    """Πολύγωνο που ορίζεται από μια λίστα κορυφών Point2D (κλείνει αυτόματα)."""

    def __init__(self, vertices, name=None):
        vertices = list(vertices)
        if not all(isinstance(v, Point2D) for v in vertices):
            raise TypeError("Όλες οι κορυφές πρέπει να είναι αντικείμενα Point2D.")
        # Αν ο χρήστης έκλεισε το πολύγωνο επαναλαμβάνοντας την πρώτη κορυφή, την αφαιρούμε
        if len(vertices) > 1 and vertices[0] == vertices[-1]:
            vertices = vertices[:-1]
        if len(vertices) < 3:
            raise ValueError("Ένα πολύγωνο χρειάζεται τουλάχιστον 3 διαφορετικές κορυφές.")
        self.vertices = vertices
        self.name = name
        if math.isclose(self.signed_area(), 0, abs_tol=TOL):
            raise ValueError("Το εμβαδόν είναι μηδέν: όλες οι κορυφές είναι συνευθειακές.")

    # ------------------------------------------------ Ειδικές μέθοδοι
    def __repr__(self):
        return f"Polygon(vertices={self.vertices!r}, name={self.name!r})"

    def __str__(self):
        label = self.name if self.name else "Πολύγωνο"
        return f"{label}: " + " → ".join(str(v) for v in self.vertices)

    def __len__(self):
        return len(self.vertices)

    def __iter__(self):
        return iter(self.vertices)

    def __getitem__(self, i):
        return self.vertices[i]

    # ------------------------------------------------ Βοηθητικά
    def _pairs(self):
        # Ζεύγη διαδοχικών κορυφών (Pi, Pi+1), μαζί με το (Pn, P1) που κλείνει το πολύγωνο
        n = len(self.vertices)
        return [(self.vertices[i], self.vertices[(i + 1) % n]) for i in range(n)]

    def type_name(self):
        n = len(self)
        return POLYGON_NAMES.get(n, f"{n}-γωνο")

    # ------------------------------------------------ Πλευρές και μήκη
    def edges(self):
        """Λίστα με τις πλευρές του πολυγώνου ως Line2D."""
        return [Line2D(p, q) for p, q in self._pairs()]

    def perimeter(self):
        return sum(e.length for e in self.edges())

    # ------------------------------------------------ Εμβαδόν και φορά
    def signed_area(self):
        """Τύπος του Gauss: θετικό για αριστερόστροφη και αρνητικό για δεξιόστροφη φορά."""
        return 0.5 * sum(p.x * q.y - q.x * p.y for p, q in self._pairs())

    def area(self):
        return abs(self.signed_area())

    def orientation(self):
        return "αριστερόστροφη (CCW)" if self.signed_area() > 0 else "δεξιόστροφη (CW)"

    def is_ccw(self):
        return self.signed_area() > 0

    # ------------------------------------------------ Κέντρο βάρους και όρια
    def centroid(self, name="G"):
        E = self.signed_area()
        cx = sum((p.x + q.x) * (p.x * q.y - q.x * p.y) for p, q in self._pairs()) / (6 * E)
        cy = sum((p.y + q.y) * (p.x * q.y - q.x * p.y) for p, q in self._pairs()) / (6 * E)
        return Point2D(cx, cy, name)

    def bounding_box(self):
        """Επιστρέφει (Xmin, Ymin, Xmax, Ymax)."""
        xs = [v.x for v in self.vertices]
        ys = [v.y for v in self.vertices]
        return min(xs), min(ys), max(xs), max(ys)

    # ------------------------------------------------ Γωνίες και μορφή
    def interior_angles(self, unit="grad"):
        """Εσωτερική γωνία σε κάθε κορυφή (ισχύει και για μη κυρτά πολύγωνα)."""
        n = len(self)
        angles = []
        for i in range(n):
            prev_v = self.vertices[i - 1]
            cur = self.vertices[i]
            next_v = self.vertices[(i + 1) % n]
            a_prev = math.atan2(prev_v.y - cur.y, prev_v.x - cur.x)
            a_next = math.atan2(next_v.y - cur.y, next_v.x - cur.x)
            diff = (a_prev - a_next) if self.is_ccw() else (a_next - a_prev)
            angles.append(convert_angle(diff % (2 * math.pi), unit))
        return angles

    def is_convex(self):
        """Κυρτό: όλα τα διαδοχικά εξωτερικά γινόμενα έχουν το ίδιο πρόσημο."""
        n = len(self)
        signs = set()
        for i in range(n):
            p0, p1, p2 = self.vertices[i], self.vertices[(i + 1) % n], self.vertices[(i + 2) % n]
            c = cross(p1.x - p0.x, p1.y - p0.y, p2.x - p1.x, p2.y - p1.y)
            if not math.isclose(c, 0, abs_tol=TOL):
                signs.add(c > 0)
        return len(signs) == 1

    def is_simple(self):
        """Απλό πολύγωνο: δεν τέμνονται πλευρές που δεν είναι διαδοχικές."""
        edges = self.edges()
        n = len(edges)
        for i in range(n):
            for j in range(i + 1, n):
                # Οι διαδοχικές πλευρές μοιράζονται κορυφή, οπότε τις παραλείπουμε
                if j == i + 1 or (i == 0 and j == n - 1):
                    continue
                if edges[i].intersection(edges[j]) is not None:
                    return False
        return True

    # ------------------------------------------------ Σημείο μέσα στο πολύγωνο
    def contains_point(self, p, include_boundary=True):
        """Ray casting. Τα σημεία πάνω στις πλευρές θεωρούνται μέσα αν include_boundary=True."""
        if any(e.contains_point(p) for e in self.edges()):
            return include_boundary
        inside = False
        for a, b in self._pairs():
            if (a.y > p.y) != (b.y > p.y):  # η πλευρά «διασχίζει» την οριζόντια ευθεία του p
                x_cross = a.x + (p.y - a.y) * (b.x - a.x) / (b.y - a.y)
                if x_cross > p.x:
                    inside = not inside
        return inside

    # ------------------------------------------------ Περιγραφή και σχεδίαση
    def describe(self, unit="grad"):
        """Επιστρέφει πλήρη περιγραφή του πολυγώνου σε κείμενο."""
        sym = "grad" if unit == "grad" else ("°" if unit == "deg" else "rad")
        xmin, ymin, xmax, ymax = self.bounding_box()
        angles = self.interior_angles(unit)
        full_turn = convert_angle(math.pi, unit)  # 200 grad ή 180°
        expected = (len(self) - 2) * full_turn
        lines = [
            f"Πολύγωνο: {self.name if self.name else '(χωρίς όνομα)'}",
            f"  Τύπος: {self.type_name()} ({len(self)} κορυφές)",
            f"  Φορά κορυφών: {self.orientation()}",
            f"  Κυρτό: {'ναι' if self.is_convex() else 'όχι'}",
            f"  Απλό (χωρίς αυτοτομές): {'ναι' if self.is_simple() else 'ΟΧΙ: οι πλευρές τέμνονται!'}",
            f"  Περίμετρος: {self.perimeter():.3f}",
            f"  Εμβαδόν: {self.area():.3f}",
            f"  Κέντρο βάρους: {self.centroid()}",
            f"  Περιβάλλον ορθογώνιο: X ∈ [{xmin:.3f}, {xmax:.3f}], Y ∈ [{ymin:.3f}, {ymax:.3f}]",
            "  Κορυφές:",
        ]
        for v, ang in zip(self.vertices, angles):
            lines.append(f"    {str(v):<28} εσωτ. γωνία = {ang:9.4f} {sym}")
        lines.append("  Πλευρές:")
        for e in self.edges():
            label = e.name if e.name else "-"
            lines.append(f"    {label:<6} μήκος = {e.length:10.3f}   αζιμούθιο = {e.azimuth():9.4f} grad")
        lines.append(f"  Άθροισμα εσωτ. γωνιών: {sum(angles):.4f} {sym}"
                     f" (θεωρητικά (n−2)·{full_turn:.0f} = {expected:.0f} {sym})")
        if not self.is_simple():
            lines.append("  ΠΡΟΣΟΧΗ: το πολύγωνο αυτοτέμνεται, οπότε το εμβαδόν, το κέντρο βάρους"
                         " και οι γωνίες δεν έχουν τη συνηθισμένη γεωμετρική σημασία.")
        return "\n".join(lines)

    def plot(self, ax=None, color="tab:green", alpha=0.3, show_labels=True, show_centroid=True):
        if ax is None:
            _, ax = plt.subplots(figsize=(6, 6))
        xs = [v.x for v in self.vertices] + [self.vertices[0].x]
        ys = [v.y for v in self.vertices] + [self.vertices[0].y]
        ax.fill(xs, ys, color=color, alpha=alpha)
        ax.plot(xs, ys, color=color, lw=1.5)
        for v in self.vertices:
            v.plot(ax, color=color, show_label=show_labels)
        if show_centroid:
            self.centroid().plot(ax, color="black", marker="x", show_label=show_labels)
        if self.name:
            ax.set_title(self.name)
        ax.set_aspect("equal")
        ax.grid(True, linestyle=":")
        ax.set_xlabel("X (Ανατολή)")
        ax.set_ylabel("Y (Βορράς)")
        return ax

### Παραδείγματα με `Polygon`

Πρώτα ένα απλό τετράπλευρο, για το οποίο υπολογίζουμε εύκολα το εμβαδόν και με το χέρι.

In [ ]:
square = Polygon([Point2D(0, 0, "1"), Point2D(10, 0, "2"),
                  Point2D(10, 10, "3"), Point2D(0, 10, "4")], name="Τετράγωνο")
print(square.describe())

Στη συνέχεια ένα **μη κυρτό** πολύγωνο (σχήμα «Γ»), δοσμένο με δεξιόστροφη φορά, και έλεγχος για το αν κάποια σημεία βρίσκονται μέσα σε αυτό.

In [ ]:
parcel = Polygon([
    Point2D(200, 500, "K1"),
    Point2D(260, 500, "K2"),
    Point2D(260, 440, "K3"),
    Point2D(320, 440, "K4"),
    Point2D(320, 400, "K5"),
    Point2D(200, 400, "K6"),
], name="Οικόπεδο σε σχήμα Γ")

print(parcel.describe(unit="deg"))

In [ ]:
test_points = [Point2D(230, 470, "Σ1"), Point2D(290, 470, "Σ2"),
               Point2D(300, 420, "Σ3"), Point2D(260, 460, "Σ4")]

for p in test_points:
    print(f"{p}: {'μέσα' if parcel.contains_point(p) else 'έξω'}")

ax = parcel.plot()
for p in test_points:
    p.plot(ax, color="tab:green" if parcel.contains_point(p) else "tab:red")
plt.show()

Τέλος, ένα πολύγωνο του οποίου οι πλευρές **αυτοτέμνονται** («παπιγιόν»). Η `describe()` το επισημαίνει, επειδή για τέτοια πολύγωνα το εμβαδόν του Gauss δεν έχει τη συνηθισμένη γεωμετρική σημασία.

In [ ]:
bowtie = Polygon([Point2D(0, 0, "A"), Point2D(6, 4, "B"),
                  Point2D(6, 0, "C"), Point2D(0, 6, "D")], name="Παπιγιόν")
print(bowtie.describe())
bowtie.plot(color="tab:red")
plt.show()

## 4. Έλεγχος λανθασμένων δεδομένων

Οι κλάσεις ελέγχουν τα δεδομένα που τους δίνουμε και εμφανίζουν κατανοητά μηνύματα σφάλματος, αντί να επιστρέφουν λάθος αποτελέσματα χωρίς προειδοποίηση.

In [ ]:
bad_inputs = [
    ("Τμήμα με ίδια άκρα",        lambda: Line2D(Point2D(1, 1), Point2D(1, 1))),
    ("Τμήμα με άκρα που δεν είναι σημεία", lambda: Line2D((0, 0), (1, 1))),
    ("Πολύγωνο με 2 κορυφές",     lambda: Polygon([Point2D(0, 0), Point2D(1, 1)])),
    ("Πολύγωνο με συνευθειακές κορυφές", lambda: Polygon([Point2D(0, 0), Point2D(1, 1), Point2D(2, 2)])),
    ("Αζιμούθιο προς το ίδιο σημείο", lambda: A.azimuth_to(A)),
]

for title, func in bad_inputs:
    try:
        func()
        print(f"{title:<36} → κανένα σφάλμα")
    except (ValueError, TypeError) as err:
        print(f"{title:<36} → {type(err).__name__}: {err}")

## 5. Όλα μαζί σε ένα σχήμα

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
plot_parcel = Polygon([Point2D(100, 180, "Π1"), Point2D(170, 190, "Π2"),
                       Point2D(180, 250, "Π3"), Point2D(120, 270, "Π4")], name="Πολύγωνο Π")
plot_parcel.plot(ax)
AB.plot(ax, color="tab:blue", arrow=True)
CD.plot(ax, color="tab:orange", arrow=True)

# Τομές των τμημάτων AB και CD με τις πλευρές του πολυγώνου
for line in (AB, CD):
    for edge in plot_parcel.edges():
        t = line.intersection(edge)
        if t is not None:
            t.plot(ax, color="black", show_label=False, marker="s")

ax.set_title("Σημεία, τμήματα και πολύγωνο")
plt.show()

## 6. Διαδραστικό playground

Σε αυτή την ενότητα έχουμε ένα απλό **διαδραστικό εργαλείο σχεδίασης** με κάναβο. Σε αυτό μπορούμε να:

- **προσθέτουμε σημεία** με κλικ,
- **ενώνουμε σημεία** με γραμμές, με έλξη (snap) πάνω σε υπάρχοντα σημεία,
- **σχεδιάζουμε πολύγωνα** και να **μετράμε το εμβαδόν** και την περίμετρό τους,
- βλέπουμε μήκη και αζιμούθια σε πραγματικό χρόνο.

### Κουμπιά

| Κουμπί | Λειτουργία |
|---|---|
| **Επιλογή** | Κλικ σε οντότητα για να δεις την πλήρη περιγραφή της (`describe()`) |
| **Σημείο** | Κλικ για νέο σημείο |
| **Γραμμή** | Κλικ στο 1ο και μετά στο 2ο σημείο |
| **Πολύγωνο** | Κλικ στις κορυφές. Κλείνει με κλικ στην 1η κορυφή, με `Enter` ή με το κουμπί «Κλείσιμο πολυγώνου» |
| **Διαγραφή** | Κλικ σε οντότητα για να διαγραφεί |
| **OSNAP: ON/OFF** | Έλξη σε υπάρχοντα **σημεία** (□), **μέσα** τμημάτων (△) και **τομές** τμημάτων (✕). Πλήκτρο `F3` |
| **SNAP GRID: ON/OFF** | Έλξη στους κόμβους του κανάβου (○), με το **βήμα** που επιλέγεις. Πλήκτρο `F9` |
| **Αναίρεση** | Ακυρώνει την τελευταία ενέργεια (`Ctrl+Z`) |
| **Εξαγωγή στην Python** | Περνά όλες τις οντότητες στη μεταβλητή `playground_data`, ως αντικείμενα `Point2D`, `Line2D` και `Polygon` |

Με τη ρόδα του ποντικιού κάνεις zoom. Με δεξί κλικ και σύρσιμο (ή `Shift` και σύρσιμο) μετακινείς την προβολή.

### Πώς συνδέεται με τις κλάσεις μας

Το εργαλείο είναι γραμμένο σε HTML και JavaScript, ώστε να τρέχει στον browser. Όταν όμως επιλέγεις μια οντότητα, ζητά την περιγραφή της **από την Python**, μέσω του μηχανισμού `google.colab.output.register_callback` του Colab. Έτσι το κείμενο που βλέπεις παράγεται από τη `describe()` των κλάσεων που φτιάξαμε παραπάνω.

**Σημαντικό:** τρέξε πρώτα όλα τα προηγούμενα κελιά, για να έχουν οριστεί οι κλάσεις.

In [ ]:
#@title Κώδικας του διαδραστικού εργαλείου (τρέξε το κελί, διπλό κλικ για να δεις τον κώδικα) { display-mode: "form" }
import uuid
from IPython.display import HTML, JSON, display

try:
    from google.colab import output as colab_output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Εδώ καταλήγουν οι οντότητες όταν πατάμε «Εξαγωγή στην Python»
playground_data = {"points": {}, "lines": {}, "polygons": {}}


def _gp_build(state):
    """Μετατρέπει την κατάσταση του εργαλείου (λεξικό από τη JavaScript) σε αντικείμενα των κλάσεών μας."""
    points = {p["name"]: Point2D(p["x"], p["y"], p["name"]) for p in state["points"]}
    lines = {l["name"]: Line2D(points[l["a"]], points[l["b"]], l["name"]) for l in state["lines"]}
    polygons = {g["name"]: Polygon([points[n] for n in g["verts"]], g["name"]) for g in state["polygons"]}
    return points, lines, polygons


def _gp_describe(state, kind, name):
    """Καλείται από τη JavaScript όταν επιλέγουμε μια οντότητα: επιστρέφει τη describe() της."""
    try:
        points, lines, polygons = _gp_build(state)
        obj = {"point": points, "line": lines, "polygon": polygons}[kind][name]
        return JSON({"ok": True, "text": obj.describe()})
    except Exception as err:
        return JSON({"ok": False, "text": f"Σφάλμα: {err}"})


def _gp_export(state):
    """Καλείται από τη JavaScript με το κουμπί «Εξαγωγή στην Python»."""
    try:
        points, lines, polygons = _gp_build(state)
    except Exception as err:
        return JSON({"ok": False, "text": f"Σφάλμα εξαγωγής: {err}"})
    # Ενημερώνουμε το ίδιο λεξικό (όχι νέο), ώστε να το βλέπουν όλα τα κελιά
    playground_data["points"] = points
    playground_data["lines"] = lines
    playground_data["polygons"] = polygons
    return JSON({"ok": True, "text": f"Εξήχθησαν στο playground_data: {len(points)} σημεία, "
                                     f"{len(lines)} γραμμές, {len(polygons)} πολύγωνα."})


if IN_COLAB:
    colab_output.register_callback("geo_playground.describe", _gp_describe)
    colab_output.register_callback("geo_playground.export", _gp_export)


def geo_playground(width=700, height=500, grid_step=1, scale=40):
    """Εμφανίζει το διαδραστικό εργαλείο.

    width, height : διαστάσεις του καμβά σε pixels
    grid_step     : αρχικό βήμα του κανάβου (μονάδες συντεταγμένων)
    scale         : αρχική κλίμακα (pixels ανά μονάδα)
    """
    html = (_PLAYGROUND_TEMPLATE
            .replace("__UID__", "gp_" + uuid.uuid4().hex[:8])
            .replace("__W__", str(int(width)))
            .replace("__H__", str(int(height)))
            .replace("__STEP__", repr(float(grid_step)).rstrip("0").rstrip("."))
            .replace("__SCALE__", str(float(scale))))
    if not IN_COLAB:
        print("Σημείωση: εκτός Colab το εργαλείο λειτουργεί, αλλά η περιγραφή από την Python "
              "και η «Εξαγωγή στην Python» δεν είναι διαθέσιμες.")
    display(HTML(html))


_PLAYGROUND_TEMPLATE = r"""
<div id="__UID__" class="gp-root">
<style>
#__UID__ { font-family: "Segoe UI", Roboto, Arial, sans-serif; font-size: 13px; color: #222; }
#__UID__ .gp-toolbar { display: flex; flex-wrap: wrap; gap: 6px; align-items: center; margin-bottom: 6px; }
#__UID__ .gp-group { display: flex; gap: 3px; padding: 3px; border: 1px solid #ccc; border-radius: 6px; background: #f7f7f7; align-items: center; }
#__UID__ button { font: inherit; padding: 4px 9px; border: 1px solid #bbb; border-radius: 4px; background: #fff; color: #222; cursor: pointer; }
#__UID__ button:hover { background: #eef3ff; }
#__UID__ button.gp-active { background: #2f6fd6; color: #fff; border-color: #2f6fd6; }
#__UID__ button.gp-on { background: #2e9e4f; color: #fff; border-color: #2e9e4f; }
#__UID__ button.gp-off { background: #e6e6e6; color: #555; }
#__UID__ select { font: inherit; padding: 3px; color: #222; background: #fff; }
#__UID__ .gp-main { display: flex; flex-wrap: wrap; gap: 8px; align-items: flex-start; }
#__UID__ canvas { border: 1px solid #999; border-radius: 4px; background: #fff; cursor: crosshair; outline: none; }
#__UID__ .gp-side { width: 300px; background: #fafafa; border: 1px solid #ccc; border-radius: 4px; padding: 6px 8px; box-sizing: border-box; }
#__UID__ .gp-side h4 { margin: 6px 0 3px; font-size: 13px; color: #2f6fd6; }
#__UID__ .gp-list { max-height: 210px; overflow-y: auto; font-family: Consolas, monospace; font-size: 12px; }
#__UID__ .gp-item { padding: 1px 3px; cursor: pointer; border-radius: 3px; white-space: nowrap; }
#__UID__ .gp-item:hover { background: #e8eefc; }
#__UID__ .gp-item.gp-sel { background: #ffe2b8; }
#__UID__ .gp-details { white-space: pre-wrap; font-family: Consolas, monospace; font-size: 11.5px; background: #fff; border: 1px solid #ddd; border-radius: 3px; padding: 5px; min-height: 60px; max-height: 260px; overflow-y: auto; margin: 0; color: #222; }
#__UID__ .gp-status { margin-top: 5px; font-family: Consolas, monospace; font-size: 12px; color: #333; background: #f1f1f1; border-radius: 4px; padding: 4px 8px; }
#__UID__ .gp-hint { margin-top: 3px; font-size: 12px; color: #666; }
</style>

<div class="gp-toolbar">
  <div class="gp-group">
    <button data-mode="select" title="Κλικ σε οντότητα για πληροφορίες">Επιλογή</button>
    <button data-mode="point" title="Προσθήκη σημείου">Σημείο</button>
    <button data-mode="line" title="Ένωση δύο σημείων">Γραμμή</button>
    <button data-mode="polygon" title="Σχεδίαση πολυγώνου και μέτρηση εμβαδού">Πολύγωνο</button>
    <button data-mode="delete" title="Διαγραφή οντότητας">Διαγραφή</button>
  </div>
  <div class="gp-group">
    <button data-toggle="osnap" title="Έλξη σε σημεία, μέσα και τομές (F3)">OSNAP</button>
    <button data-toggle="grid" title="Έλξη στον κάναβο (F9)">SNAP GRID</button>
    <span>&nbsp;Βήμα:</span>
    <select class="gp-step">
      <option value="0.1">0.1</option><option value="0.5">0.5</option><option value="1">1</option>
      <option value="2">2</option><option value="5">5</option><option value="10">10</option>
    </select>
  </div>
  <div class="gp-group">
    <button data-action="close" title="Κλείσιμο του πολυγώνου που σχεδιάζεται (Enter)">Κλείσιμο πολυγώνου</button>
    <button data-action="undo" title="Αναίρεση (Ctrl+Z)">Αναίρεση</button>
    <button data-action="clear" title="Διαγραφή όλων">Καθαρισμός</button>
    <button data-action="view" title="Επαναφορά zoom και θέσης">Αρχική προβολή</button>
    <button data-action="export" title="Μεταφορά των οντοτήτων στη μεταβλητή playground_data της Python">Εξαγωγή στην Python</button>
  </div>
</div>

<div class="gp-main">
  <canvas tabindex="0"></canvas>
  <div class="gp-side">
    <h4>Σημεία <span class="gp-n-points"></span></h4>
    <div class="gp-list gp-points"></div>
    <h4>Γραμμές <span class="gp-n-lines"></span></h4>
    <div class="gp-list gp-lines"></div>
    <h4>Πολύγωνα <span class="gp-n-polygons"></span></h4>
    <div class="gp-list gp-polygons"></div>
    <h4>Περιγραφή επιλεγμένης οντότητας</h4>
    <pre class="gp-details">Επίλεξε μια οντότητα (λειτουργία «Επιλογή» ή κλικ στη λίστα).</pre>
  </div>
</div>
<div class="gp-status">&nbsp;</div>
<div class="gp-hint">Ρόδα ποντικιού: zoom · Δεξί κλικ + σύρσιμο ή Shift + σύρσιμο: μετακίνηση · Esc: ακύρωση σχεδίασης · F3: OSNAP · F9: SNAP GRID</div>

<script>
(function () {
  const root = document.getElementById("__UID__");
  const canvas = root.querySelector("canvas");
  const ctx = canvas.getContext("2d");
  const W = __W__, H = __H__;
  const INIT_SCALE = __SCALE__;
  const dpr = window.devicePixelRatio || 1;
  canvas.width = W * dpr; canvas.height = H * dpr;
  canvas.style.width = W + "px"; canvas.style.height = H + "px";
  ctx.setTransform(dpr, 0, 0, dpr, 0, 0);

  const HAS_PY = (typeof google !== "undefined" && google.colab && google.colab.kernel &&
                  typeof google.colab.kernel.invokeFunction === "function");

  // ------------------------------------------------------------ Κατάσταση
  const S = { points: [], lines: [], polygons: [], nextP: 1, nextL: 1, nextG: 1 };
  const history = [];
  let mode = "point";
  let osnap = true, gridSnap = true, gridStep = __STEP__;
  let view = { scale: INIT_SCALE, ox: W / 2, oy: H / 2 };
  let mouse = null, snap = null, draft = [], selected = null, panning = null;
  let message = "";
  let lastDetailsKey = null;

  const SNAP_NAMES = { endpoint: "Σημείο", midpoint: "Μέσο", intersection: "Τομή", grid: "Κάναβος", free: "Ελεύθερα" };
  const MODE_NAMES = { select: "Επιλογή", point: "Σημείο", line: "Γραμμή", polygon: "Πολύγωνο", delete: "Διαγραφή" };
  const MODE_HINTS = {
    select: "Κλικ σε οντότητα για να δεις την περιγραφή της",
    point: "Κλικ για νέο σημείο",
    line: "Κλικ στο 1ο και μετά στο 2ο σημείο",
    polygon: "Κλικ στις κορυφές · κλείσιμο: κλικ στην 1η κορυφή ή Enter · Backspace: αφαίρεση τελευταίας κορυφής",
    delete: "Κλικ σε σημείο, γραμμή ή πολύγωνο για διαγραφή",
  };

  // ------------------------------------------------------------ Βοηθητικά
  const clean = v => Math.round(v * 1e9) / 1e9;
  const fmt = (v, d = 3) => (Math.abs(v) < 1e-12 ? 0 : v).toFixed(d);
  const toWorld = (sx, sy) => ({ x: (sx - view.ox) / view.scale, y: (view.oy - sy) / view.scale });
  const toScreen = (x, y) => ({ sx: view.ox + x * view.scale, sy: view.oy - y * view.scale });
  const P = id => S.points.find(p => p.id === id);

  function azimuthGrad(a, b) {
    let t = Math.atan2(b.x - a.x, b.y - a.y);   // από τον Βορρά, δεξιόστροφα
    if (t < 0) t += 2 * Math.PI;
    return t * 200 / Math.PI;
  }
  function elevationGrad(a, b) {
    let t = Math.atan2(b.y - a.y, b.x - a.x);   // από τον άξονα X, αριστερόστροφα
    if (t < 0) t += 2 * Math.PI;
    return t * 200 / Math.PI;
  }
  const dist = (a, b) => Math.hypot(b.x - a.x, b.y - a.y);
  function signedArea(pts) {
    let s = 0;
    for (let i = 0; i < pts.length; i++) {
      const p = pts[i], q = pts[(i + 1) % pts.length];
      s += p.x * q.y - q.x * p.y;
    }
    return s / 2;
  }
  function perimeter(pts) {
    let s = 0;
    for (let i = 0; i < pts.length; i++) s += dist(pts[i], pts[(i + 1) % pts.length]);
    return s;
  }
  function centroid(pts) {
    const A = signedArea(pts);
    let cx = 0, cy = 0;
    for (let i = 0; i < pts.length; i++) {
      const p = pts[i], q = pts[(i + 1) % pts.length], c = p.x * q.y - q.x * p.y;
      cx += (p.x + q.x) * c; cy += (p.y + q.y) * c;
    }
    return Math.abs(A) < 1e-12 ? pts[0] : { x: cx / (6 * A), y: cy / (6 * A) };
  }
  function segInter(p1, p2, p3, p4) {
    const rx = p2.x - p1.x, ry = p2.y - p1.y, sx = p4.x - p3.x, sy = p4.y - p3.y;
    const d = rx * sy - ry * sx;
    if (Math.abs(d) < 1e-12) return null;
    const qx = p3.x - p1.x, qy = p3.y - p1.y;
    const t = (qx * sy - qy * sx) / d, u = (qx * ry - qy * rx) / d;
    if (t < -1e-9 || t > 1 + 1e-9 || u < -1e-9 || u > 1 + 1e-9) return null;
    return { x: p1.x + t * rx, y: p1.y + t * ry };
  }
  function distToSeg(px, py, ax, ay, bx, by) {
    const dx = bx - ax, dy = by - ay, L2 = dx * dx + dy * dy;
    let t = L2 ? ((px - ax) * dx + (py - ay) * dy) / L2 : 0;
    t = Math.max(0, Math.min(1, t));
    return Math.hypot(px - (ax + t * dx), py - (ay + t * dy));
  }
  function pointInPoly(pt, pts) {
    let inside = false;
    for (let i = 0, j = pts.length - 1; i < pts.length; j = i++) {
      const a = pts[i], b = pts[j];
      if ((a.y > pt.y) !== (b.y > pt.y) && pt.x < (b.x - a.x) * (pt.y - a.y) / (b.y - a.y) + a.x) inside = !inside;
    }
    return inside;
  }
  const polyPts = g => g.verts.map(P);
  function segments() {
    const segs = [];
    S.lines.forEach(l => segs.push([P(l.a), P(l.b)]));
    S.polygons.forEach(g => g.verts.forEach((v, i) => segs.push([P(v), P(g.verts[(i + 1) % g.verts.length])])));
    return segs;
  }

  // ------------------------------------------------------------ Snap
  function findSnap(sx, sy) {
    const tol = 10;
    let best = null;
    const consider = (x, y, type, pid, prio) => {
      const s = toScreen(x, y), d = Math.hypot(s.sx - sx, s.sy - sy);
      if (d <= tol && (!best || prio < best.prio || (prio === best.prio && d < best.d)))
        best = { x, y, type, pid, prio, d };
    };
    if (osnap) {
      S.points.forEach(p => consider(p.x, p.y, "endpoint", p.id, 0));
      const segs = segments();
      for (let i = 0; i < segs.length; i++)
        for (let j = i + 1; j < segs.length; j++) {
          const q = segInter(segs[i][0], segs[i][1], segs[j][0], segs[j][1]);
          if (q) consider(q.x, q.y, "intersection", null, 1);
        }
      segs.forEach(([a, b]) => consider((a.x + b.x) / 2, (a.y + b.y) / 2, "midpoint", null, 2));
    }
    if (best) return best;
    const w = toWorld(sx, sy);
    if (gridSnap)
      return { x: clean(Math.round(w.x / gridStep) * gridStep), y: clean(Math.round(w.y / gridStep) * gridStep), type: "grid" };
    return { x: w.x, y: w.y, type: "free" };
  }
  function existingAt(s) {
    if (s.pid) return s.pid;
    const p = S.points.find(p => Math.abs(p.x - s.x) < 1e-9 && Math.abs(p.y - s.y) < 1e-9);
    return p ? p.id : null;
  }
  function pointAt(s) {
    const id = existingAt(s);
    if (id) return id;
    const p = { id: "p" + S.nextP, name: "P" + S.nextP, x: clean(s.x), y: clean(s.y) };
    S.nextP++;
    S.points.push(p);
    return p.id;
  }

  // ------------------------------------------------------------ Ενέργειες
  function act(fn) {
    const before = JSON.stringify(S);
    fn();
    if (JSON.stringify(S) !== before) { history.push(before); if (history.length > 200) history.shift(); }
    refresh();
  }
  function undo() {
    if (!history.length) { message = "Δεν υπάρχει κάτι για αναίρεση."; refresh(); return; }
    Object.assign(S, JSON.parse(history.pop()));
    draft = draft.filter(id => P(id));
    if (selected && !findEntity(selected)) selected = null;
    message = "Αναίρεση.";
    refresh();
  }
  function closePolygon() {
    if (draft.length < 3) { message = "Το πολύγωνο χρειάζεται τουλάχιστον 3 κορυφές."; return; }
    const pts = draft.map(P);
    if (Math.abs(signedArea(pts)) < 1e-12) { message = "Οι κορυφές είναι συνευθειακές: το εμβαδόν είναι μηδέν."; return; }
    const g = { id: "g" + S.nextG, name: "Π" + S.nextG, verts: draft.slice() };
    S.nextG++;
    S.polygons.push(g);
    selected = { kind: "polygon", id: g.id };
    message = `Δημιουργήθηκε το ${g.name}: εμβαδόν ${fmt(Math.abs(signedArea(pts)))}`;
    draft = [];
  }
  function findEntity(h) {
    const arr = { point: S.points, line: S.lines, polygon: S.polygons }[h.kind];
    return arr.find(e => e.id === h.id);
  }
  function removeEntity(h) {
    if (h.kind === "point") {
      S.points = S.points.filter(p => p.id !== h.id);
      S.lines = S.lines.filter(l => l.a !== h.id && l.b !== h.id);
      S.polygons = S.polygons.filter(g => !g.verts.includes(h.id));
      draft = draft.filter(id => id !== h.id);
    } else if (h.kind === "line") {
      S.lines = S.lines.filter(l => l.id !== h.id);
    } else {
      S.polygons = S.polygons.filter(g => g.id !== h.id);
    }
    if (selected && !findEntity(selected)) selected = null;
  }
  function hit(sx, sy) {
    const tol = 8;
    let best = null, bd = tol;
    S.points.forEach(p => {
      const s = toScreen(p.x, p.y), d = Math.hypot(s.sx - sx, s.sy - sy);
      if (d <= bd) { bd = d; best = { kind: "point", id: p.id }; }
    });
    if (best) return best;
    bd = tol;
    S.lines.forEach(l => {
      const a = toScreen(P(l.a).x, P(l.a).y), b = toScreen(P(l.b).x, P(l.b).y);
      const d = distToSeg(sx, sy, a.sx, a.sy, b.sx, b.sy);
      if (d <= bd) { bd = d; best = { kind: "line", id: l.id }; }
    });
    if (best) return best;
    const w = toWorld(sx, sy);
    let bestArea = Infinity;
    S.polygons.forEach(g => {
      const pts = polyPts(g);
      let near = false;
      for (let i = 0; i < pts.length; i++) {
        const a = toScreen(pts[i].x, pts[i].y), b = toScreen(pts[(i + 1) % pts.length].x, pts[(i + 1) % pts.length].y);
        if (distToSeg(sx, sy, a.sx, a.sy, b.sx, b.sy) <= tol) near = true;
      }
      const A = Math.abs(signedArea(pts));
      if ((near || pointInPoly(w, pts)) && A < bestArea) { bestArea = A; best = { kind: "polygon", id: g.id }; }
    });
    return best;
  }

  function handleClick(sx, sy) {
    message = "";
    act(() => {
      if (mode === "point") {
        const id = pointAt(findSnap(sx, sy));
        selected = { kind: "point", id };
      } else if (mode === "line") {
        const id = pointAt(findSnap(sx, sy));
        if (!draft.length) { draft = [id]; return; }
        if (id === draft[0]) return;
        const a = draft[0];
        const exists = S.lines.find(l => (l.a === a && l.b === id) || (l.a === id && l.b === a));
        if (exists) { message = `Η γραμμή ${exists.name} υπάρχει ήδη.`; }
        else {
          const l = { id: "l" + S.nextL, name: P(a).name + P(id).name, a, b: id };
          S.nextL++;
          S.lines.push(l);
          selected = { kind: "line", id: l.id };
          message = `Δημιουργήθηκε η ${l.name}: μήκος ${fmt(dist(P(a), P(id)))}`;
        }
        draft = [];
      } else if (mode === "polygon") {
        const s = findSnap(sx, sy);
        const ex = existingAt(s);
        if (draft.length >= 3 && ex === draft[0]) { closePolygon(); return; }
        const id = pointAt(s);
        if (draft.includes(id)) { message = "Η κορυφή υπάρχει ήδη στο πολύγωνο."; return; }
        draft.push(id);
      } else if (mode === "delete") {
        const h = hit(sx, sy);
        if (h) { message = `Διαγράφηκε: ${findEntity(h).name}`; removeEntity(h); }
      } else if (mode === "select") {
        selected = hit(sx, sy);
      }
    });
  }

  // ------------------------------------------------------------ Σχεδίαση
  function niceGrid() {
    let step = gridStep;
    const mult = [2, 2.5, 2];
    let i = 0;
    while (step * view.scale < 14) { step *= mult[i % 3]; i++; }
    return step;
  }
  function drawGrid() {
    const step = niceGrid();
    const tl = toWorld(0, 0), br = toWorld(W, H);
    const x0 = Math.floor(tl.x / step), x1 = Math.ceil(br.x / step);
    const y0 = Math.floor(br.y / step), y1 = Math.ceil(tl.y / step);
    ctx.lineWidth = 1;
    ctx.font = "10px Consolas, monospace";
    ctx.fillStyle = "#888";
    for (let i = x0; i <= x1; i++) {
      const sx = Math.round(toScreen(i * step, 0).sx) + 0.5;
      const major = i % 5 === 0;
      ctx.strokeStyle = major ? "#cfd6e0" : "#eceff3";
      ctx.beginPath(); ctx.moveTo(sx, 0); ctx.lineTo(sx, H); ctx.stroke();
      if (major) ctx.fillText(+(i * step).toFixed(4), sx + 2, H - 4);
    }
    for (let j = y0; j <= y1; j++) {
      const sy = Math.round(toScreen(0, j * step).sy) + 0.5;
      const major = j % 5 === 0;
      ctx.strokeStyle = major ? "#cfd6e0" : "#eceff3";
      ctx.beginPath(); ctx.moveTo(0, sy); ctx.lineTo(W, sy); ctx.stroke();
      if (major) ctx.fillText(+(j * step).toFixed(4), 3, sy - 3);
    }
    // Άξονες
    const o = toScreen(0, 0);
    ctx.strokeStyle = "#7a8594"; ctx.lineWidth = 1.5;
    ctx.beginPath(); ctx.moveTo(0, o.sy); ctx.lineTo(W, o.sy); ctx.moveTo(o.sx, 0); ctx.lineTo(o.sx, H); ctx.stroke();
    ctx.fillStyle = "#4a5563"; ctx.font = "bold 12px Arial";
    ctx.fillText("X", W - 14, Math.min(Math.max(o.sy - 6, 14), H - 18));
    ctx.fillText("Y (Βορράς)", Math.min(Math.max(o.sx + 6, 4), W - 70), 14);
  }
  function label(text, sx, sy, color = "#222", bg = "rgba(255,255,255,0.8)") {
    ctx.font = "12px Arial";
    const w = ctx.measureText(text).width;
    sx = Math.min(Math.max(sx, 3), W - w - 3);     // να μη βγαίνει έξω από τον καμβά
    sy = Math.min(Math.max(sy, 14), H - 4);
    ctx.fillStyle = bg; ctx.fillRect(sx - 2, sy - 11, w + 4, 15);
    ctx.fillStyle = color; ctx.fillText(text, sx, sy);
  }
  const isSel = (kind, id) => selected && selected.kind === kind && selected.id === id;

  function draw() {
    ctx.clearRect(0, 0, W, H);
    ctx.fillStyle = "#fff"; ctx.fillRect(0, 0, W, H);
    drawGrid();

    // Πολύγωνα
    S.polygons.forEach(g => {
      const pts = polyPts(g), sel = isSel("polygon", g.id);
      ctx.beginPath();
      pts.forEach((p, i) => { const s = toScreen(p.x, p.y); i ? ctx.lineTo(s.sx, s.sy) : ctx.moveTo(s.sx, s.sy); });
      ctx.closePath();
      ctx.fillStyle = sel ? "rgba(255,150,30,0.30)" : "rgba(46,158,79,0.22)";
      ctx.fill();
      ctx.strokeStyle = sel ? "#e07b00" : "#2e9e4f"; ctx.lineWidth = sel ? 2.5 : 1.8; ctx.stroke();
      const c = centroid(pts), s = toScreen(c.x, c.y);
      label(`${g.name}  E=${fmt(Math.abs(signedArea(pts)))}`, s.sx - 30, s.sy, sel ? "#a05500" : "#1d6b35");
    });
    // Γραμμές
    S.lines.forEach(l => {
      const a = P(l.a), b = P(l.b), sa = toScreen(a.x, a.y), sb = toScreen(b.x, b.y), sel = isSel("line", l.id);
      ctx.strokeStyle = sel ? "#e07b00" : "#2f6fd6"; ctx.lineWidth = sel ? 3 : 2;
      ctx.beginPath(); ctx.moveTo(sa.sx, sa.sy); ctx.lineTo(sb.sx, sb.sy); ctx.stroke();
      label(`${l.name} ${fmt(dist(a, b))}`, (sa.sx + sb.sx) / 2 + 4, (sa.sy + sb.sy) / 2 - 4, sel ? "#a05500" : "#1f4f9e");
    });
    // Σημεία
    S.points.forEach(p => {
      const s = toScreen(p.x, p.y), sel = isSel("point", p.id), inDraft = draft.includes(p.id);
      ctx.beginPath(); ctx.arc(s.sx, s.sy, sel ? 5.5 : 4, 0, 2 * Math.PI);
      ctx.fillStyle = sel ? "#e07b00" : (inDraft ? "#7b2fd6" : "#c62828"); ctx.fill();
      ctx.font = "bold 12px Arial"; ctx.fillStyle = "#222"; ctx.fillText(p.name, s.sx + 6, s.sy - 6);
    });

    // Σχέδιο σε εξέλιξη (rubber band)
    const cur = (snap && mouse) ? snap : null;
    if (mode === "line" && draft.length === 1 && cur) {
      const a = P(draft[0]), sa = toScreen(a.x, a.y), sc = toScreen(cur.x, cur.y);
      ctx.setLineDash([6, 4]); ctx.strokeStyle = "#2f6fd6"; ctx.lineWidth = 1.5;
      ctx.beginPath(); ctx.moveTo(sa.sx, sa.sy); ctx.lineTo(sc.sx, sc.sy); ctx.stroke(); ctx.setLineDash([]);
      if (dist(a, cur) > 1e-12)
        label(`S=${fmt(dist(a, cur))}  α=${fmt(azimuthGrad(a, cur), 4)}g`, sc.sx + 12, sc.sy + 22, "#1f4f9e");
    }
    if (mode === "polygon" && draft.length) {
      const pts = draft.map(P);
      const all = cur ? pts.concat([cur]) : pts;
      ctx.beginPath();
      all.forEach((p, i) => { const s = toScreen(p.x, p.y); i ? ctx.lineTo(s.sx, s.sy) : ctx.moveTo(s.sx, s.sy); });
      if (all.length >= 3) {
        ctx.closePath();
        ctx.fillStyle = "rgba(123,47,214,0.12)"; ctx.fill();
      }
      ctx.setLineDash([6, 4]); ctx.strokeStyle = "#7b2fd6"; ctx.lineWidth = 1.5; ctx.stroke(); ctx.setLineDash([]);
      if (all.length >= 3 && cur) {
        const sc = toScreen(cur.x, cur.y);
        label(`E=${fmt(Math.abs(signedArea(all)))}  Περ.=${fmt(perimeter(all))}`, sc.sx + 12, sc.sy + 22, "#5a1fa0");
      }
    }

    // Δείκτης snap
    if (cur && mode !== "select" && mode !== "delete") {
      const s = toScreen(cur.x, cur.y);
      ctx.strokeStyle = "#d000c8"; ctx.lineWidth = 2;
      ctx.beginPath();
      if (cur.type === "endpoint") ctx.rect(s.sx - 6, s.sy - 6, 12, 12);
      else if (cur.type === "midpoint") { ctx.moveTo(s.sx, s.sy - 7); ctx.lineTo(s.sx + 7, s.sy + 5); ctx.lineTo(s.sx - 7, s.sy + 5); ctx.closePath(); }
      else if (cur.type === "intersection") { ctx.moveTo(s.sx - 6, s.sy - 6); ctx.lineTo(s.sx + 6, s.sy + 6); ctx.moveTo(s.sx + 6, s.sy - 6); ctx.lineTo(s.sx - 6, s.sy + 6); }
      else if (cur.type === "grid") { ctx.arc(s.sx, s.sy, 4, 0, 2 * Math.PI); }
      else { ctx.moveTo(s.sx - 5, s.sy); ctx.lineTo(s.sx + 5, s.sy); ctx.moveTo(s.sx, s.sy - 5); ctx.lineTo(s.sx, s.sy + 5); }
      ctx.stroke();
      if (cur.type !== "free") label(SNAP_NAMES[cur.type], s.sx + 10, s.sy - 10, "#9a0094");
    }
  }

  // ------------------------------------------------------------ Πλευρικό πάνελ
  function stateForPython() {
    const name = id => P(id).name;
    return {
      points: S.points.map(p => ({ name: p.name, x: p.x, y: p.y })),
      lines: S.lines.map(l => ({ name: l.name, a: name(l.a), b: name(l.b) })),
      polygons: S.polygons.map(g => ({ name: g.name, verts: g.verts.map(name) })),
    };
  }
  function jsDescribe(h) {
    const e = findEntity(h);
    if (h.kind === "point")
      return `Σημείο: ${e.name}\n  X = ${fmt(e.x)}\n  Y = ${fmt(e.y)}`;
    if (h.kind === "line") {
      const a = P(e.a), b = P(e.b);
      return `Ευθύγραμμο τμήμα: ${e.name}\n  Αρχή : ${a.name}(${fmt(a.x)}, ${fmt(a.y)})\n  Τέλος: ${b.name}(${fmt(b.x)}, ${fmt(b.y)})\n` +
             `  ΔX = ${fmt(b.x - a.x)},  ΔY = ${fmt(b.y - a.y)}\n  Μήκος: ${fmt(dist(a, b))}\n` +
             `  Αζιμούθιο: ${fmt(azimuthGrad(a, b), 4)} grad\n  Γωνία με τον άξονα X (γωνία ύψους): ${fmt(elevationGrad(a, b), 4)} grad`;
    }
    const pts = polyPts(e), A = signedArea(pts), c = centroid(pts);
    return `Πολύγωνο: ${e.name}\n  Κορυφές: ${pts.map(p => p.name).join(" → ")}\n  Φορά: ${A > 0 ? "αριστερόστροφη (CCW)" : "δεξιόστροφη (CW)"}\n` +
           `  Περίμετρος: ${fmt(perimeter(pts))}\n  Εμβαδόν: ${fmt(Math.abs(A))}\n  Κέντρο βάρους: (${fmt(c.x)}, ${fmt(c.y)})`;
  }
  async function updateDetails() {
    const box = root.querySelector(".gp-details");
    const key = selected ? JSON.stringify([selected, stateForPython()]) : null;
    if (key === lastDetailsKey) return;
    lastDetailsKey = key;
    if (!selected) { box.textContent = "Επίλεξε μια οντότητα (λειτουργία «Επιλογή» ή κλικ στη λίστα)."; return; }
    const e = findEntity(selected);
    if (!HAS_PY) { box.textContent = jsDescribe(selected) + "\n\n(Η πλήρης describe() της Python είναι διαθέσιμη μόνο στο Colab.)"; return; }
    box.textContent = "Υπολογισμός από την Python…";
    try {
      const res = await google.colab.kernel.invokeFunction("geo_playground.describe", [stateForPython(), selected.kind, e.name], {});
      if (key !== lastDetailsKey) return;
      box.textContent = res.data["application/json"].text;
    } catch (err) {
      box.textContent = jsDescribe(selected) + "\n\n(Η Python δεν απάντησε: " + err + ")";
    }
  }
  function renderLists() {
    const mk = (kind, e, text) =>
      `<div class="gp-item${isSel(kind, e.id) ? " gp-sel" : ""}" data-kind="${kind}" data-id="${e.id}">${text}</div>`;
    root.querySelector(".gp-points").innerHTML =
      S.points.map(p => mk("point", p, `${p.name.padEnd(4)} (${fmt(p.x)}, ${fmt(p.y)})`)).join("");
    root.querySelector(".gp-lines").innerHTML =
      S.lines.map(l => { const a = P(l.a), b = P(l.b);
        return mk("line", l, `${l.name.padEnd(6)} S=${fmt(dist(a, b))}  α=${fmt(azimuthGrad(a, b), 4)}g`); }).join("");
    root.querySelector(".gp-polygons").innerHTML =
      S.polygons.map(g => { const pts = polyPts(g);
        return mk("polygon", g, `${g.name.padEnd(4)} E=${fmt(Math.abs(signedArea(pts)))}  Περ.=${fmt(perimeter(pts))}`); }).join("");
    root.querySelector(".gp-n-points").textContent = `(${S.points.length})`;
    root.querySelector(".gp-n-lines").textContent = `(${S.lines.length})`;
    const total = S.polygons.reduce((s, g) => s + Math.abs(signedArea(polyPts(g))), 0);
    root.querySelector(".gp-n-polygons").textContent = `(${S.polygons.length})` + (S.polygons.length ? ` · ΣE=${fmt(total)}` : "");
  }
  function renderToolbar() {
    root.querySelectorAll("[data-mode]").forEach(b => b.classList.toggle("gp-active", b.dataset.mode === mode));
    const t = (name, on) => {
      const b = root.querySelector(`[data-toggle="${name}"]`);
      b.classList.toggle("gp-on", on); b.classList.toggle("gp-off", !on);
      b.textContent = (name === "osnap" ? "OSNAP" : "SNAP GRID") + (on ? ": ON" : ": OFF");
    };
    t("osnap", osnap); t("grid", gridSnap);
  }
  function renderStatus() {
    let txt;
    if (mouse) {
      const w = toWorld(mouse.sx, mouse.sy), c = snap || w;
      txt = `X=${fmt(c.x)}  Y=${fmt(c.y)}`;
      if (snap) txt += `  [${SNAP_NAMES[snap.type]}]`;
    } else txt = "X=-  Y=-";
    txt += `  |  ${MODE_NAMES[mode]}: ${MODE_HINTS[mode]}`;
    if (message) txt += `  |  ${message}`;
    root.querySelector(".gp-status").textContent = txt;
  }
  function refresh() {
    if (mouse && ["point", "line", "polygon"].includes(mode)) snap = findSnap(mouse.sx, mouse.sy); else snap = null;
    draw(); renderLists(); renderToolbar(); renderStatus(); updateDetails();
  }

  // ------------------------------------------------------------ Συμβάντα
  root.querySelectorAll("[data-mode]").forEach(b => b.addEventListener("click", () => {
    mode = b.dataset.mode; draft = []; message = ""; refresh(); canvas.focus();
  }));
  root.querySelector('[data-toggle="osnap"]').addEventListener("click", () => { osnap = !osnap; refresh(); canvas.focus(); });
  root.querySelector('[data-toggle="grid"]').addEventListener("click", () => { gridSnap = !gridSnap; refresh(); canvas.focus(); });
  const stepSel = root.querySelector(".gp-step");
  stepSel.value = String(gridStep);
  if (stepSel.value !== String(gridStep)) {
    const o = document.createElement("option"); o.value = o.textContent = String(gridStep); stepSel.appendChild(o); stepSel.value = String(gridStep);
  }
  stepSel.addEventListener("change", () => { gridStep = parseFloat(stepSel.value); refresh(); });
  root.querySelector('[data-action="close"]').addEventListener("click", () => {
    if (mode !== "polygon") { message = "Η εντολή αφορά τη λειτουργία «Πολύγωνο»."; refresh(); return; }
    act(closePolygon);
  });
  root.querySelector('[data-action="undo"]').addEventListener("click", undo);
  root.querySelector('[data-action="clear"]').addEventListener("click", () => {
    if (!S.points.length) return;
    act(() => { S.points = []; S.lines = []; S.polygons = []; S.nextP = 1; S.nextL = 1; S.nextG = 1; draft = []; selected = null; });
    message = "Όλα διαγράφηκαν (μπορείς να τα επαναφέρεις με «Αναίρεση»).";
    renderStatus();
  });
  root.querySelector('[data-action="view"]').addEventListener("click", () => {
    view = { scale: INIT_SCALE, ox: W / 2, oy: H / 2 }; refresh();
  });
  root.querySelector('[data-action="export"]').addEventListener("click", async () => {
    if (!HAS_PY) {
      message = "Η εξαγωγή λειτουργεί μόνο στο Colab. Τα δεδομένα σε JSON εμφανίζονται στην περιγραφή.";
      lastDetailsKey = "__export__";
      root.querySelector(".gp-details").textContent = JSON.stringify(stateForPython(), null, 1);
      refresh(); return;
    }
    message = "Εξαγωγή…"; renderStatus();
    try {
      const res = await google.colab.kernel.invokeFunction("geo_playground.export", [stateForPython()], {});
      message = res.data["application/json"].text;
    } catch (err) { message = "Σφάλμα εξαγωγής: " + err; }
    renderStatus();
  });
  root.querySelector(".gp-side").addEventListener("click", ev => {
    const it = ev.target.closest(".gp-item");
    if (!it) return;
    selected = { kind: it.dataset.kind, id: it.dataset.id };
    refresh();
  });

  canvas.addEventListener("contextmenu", e => e.preventDefault());
  canvas.addEventListener("mousedown", e => {
    canvas.focus();
    if (e.button === 2 || e.button === 1 || e.shiftKey) {
      panning = { sx: e.offsetX, sy: e.offsetY, ox: view.ox, oy: view.oy };
      e.preventDefault(); return;
    }
    if (e.button === 0) handleClick(e.offsetX, e.offsetY);
  });
  canvas.addEventListener("mousemove", e => {
    mouse = { sx: e.offsetX, sy: e.offsetY };
    if (panning) { view.ox = panning.ox + e.offsetX - panning.sx; view.oy = panning.oy + e.offsetY - panning.sy; }
    if (["point", "line", "polygon"].includes(mode)) snap = findSnap(mouse.sx, mouse.sy); else snap = null;
    draw(); renderStatus();
  });
  window.addEventListener("mouseup", () => { panning = null; });
  canvas.addEventListener("mouseleave", () => { mouse = null; snap = null; draw(); renderStatus(); });
  canvas.addEventListener("wheel", e => {
    e.preventDefault();
    const f = e.deltaY < 0 ? 1.15 : 1 / 1.15;
    const ns = Math.min(5000, Math.max(0.5, view.scale * f));
    const w = toWorld(e.offsetX, e.offsetY);
    view.scale = ns;
    view.ox = e.offsetX - w.x * ns; view.oy = e.offsetY + w.y * ns;
    refresh();
  }, { passive: false });
  canvas.addEventListener("keydown", e => {
    if (e.key === "Escape") { draft = []; message = "Ακύρωση."; }
    else if (e.key === "Enter") { if (mode === "polygon") act(closePolygon); }
    else if (e.key === "Backspace") { draft.pop(); }
    else if (e.key === "F3") { osnap = !osnap; }
    else if (e.key === "F9") { gridSnap = !gridSnap; }
    else if ((e.ctrlKey || e.metaKey) && e.key.toLowerCase() === "z") { undo(); e.preventDefault(); return; }
    else return;
    e.preventDefault();
    refresh();
  });

  refresh();
})();
</script>
</div>
"""

print("Το διαδραστικό εργαλείο είναι έτοιμο. Τρέξε το επόμενο κελί.")

### Άνοιγμα του εργαλείου

Δοκίμασε για παράδειγμα:
1. Με ενεργό το **SNAP GRID**, σχεδίασε ένα ορθογώνιο $4 \times 3$ στη λειτουργία **Πολύγωνο**. Το εμβαδόν πρέπει να βγει $12$.
2. Στη λειτουργία **Γραμμή**, ένωσε τις απέναντι κορυφές του (διαγώνιοι), με ενεργό το **OSNAP**.
3. Στη λειτουργία **Σημείο**, πλησίασε το σημείο όπου τέμνονται οι διαγώνιοι: εμφανίζεται ο δείκτης «Τομή». Κάνε κλικ για να δημιουργήσεις σημείο ακριβώς εκεί.
4. Στη λειτουργία **Επιλογή**, κάνε κλικ στο πολύγωνο για να δεις την πλήρη περιγραφή του.

In [ ]:
geo_playground(width=700, height=500, grid_step=1, scale=40)

### Χρήση των δεδομένων του εργαλείου στην Python

Αφού πατήσεις **«Εξαγωγή στην Python»** στο εργαλείο, τρέξε το παρακάτω κελί. Οι οντότητες είναι πλέον κανονικά αντικείμενα των κλάσεών μας, οπότε μπορούμε να κάνουμε με αυτά ό,τι και στις προηγούμενες ενότητες.

In [ ]:
pts = playground_data["points"]
lns = playground_data["lines"]
pols = playground_data["polygons"]

if not (pts or lns or pols):
    print("Δεν υπάρχουν δεδομένα. Σχεδίασε κάτι στο εργαλείο και πάτα «Εξαγωγή στην Python».")
else:
    print(f"Σημεία: {list(pts)}")
    print(f"Γραμμές: {list(lns)}")
    print(f"Πολύγωνα: {list(pols)}")
    print()
    for poly in pols.values():
        print(poly.describe())
        print()

    # Σχεδίαση με matplotlib, με τις ίδιες μεθόδους plot() των κλάσεων
    fig, ax = plt.subplots(figsize=(6, 6))
    for poly in pols.values():
        poly.plot(ax)
    for line in lns.values():
        line.plot(ax)
    for p in pts.values():
        p.plot(ax)
    ax.set_title("Οντότητες από το διαδραστικό εργαλείο")
    plt.show()